In [ ]:
!pip install sentence-transformers
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

In [ ]:
BASE_PATH = "/content/drive/MyDrive/AI-Placement-Copilot-ML"

resume_path = os.path.join(
    BASE_PATH,
    "datasets/processed/processed_resumes.csv"
)

job_path = os.path.join(
    BASE_PATH,
    "processed/clean_jobs.csv"
)

output_path = os.path.join(
    BASE_PATH,
    "outputs"
)

In [ ]:
df_resume = pd.read_csv(resume_path)
df_jobs = pd.read_csv(job_path)

print("Total Resumes :", len(df_resume))
print("Total Jobs :", len(df_jobs))

Resumes: 224
Jobs: 9380


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence Transformer Loaded Successfully!")

In [ ]:
resume_embeddings = model.encode(
    df_resume["clean_text"].tolist(),
    show_progress_bar=True
)

print("Resume Embeddings Shape:", resume_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
job_embeddings = model.encode(
    df_jobs["job_summary"].tolist(),
    show_progress_bar=True
)

print("Job Embeddings Shape:", job_embeddings.shape)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
dimension = job_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(job_embeddings).astype("float32")
)

print("FAISS Index Created Successfully!")

Batches:   0%|          | 0/294 [00:00<?, ?it/s]

In [ ]:
all_matches = []

for i in range(len(df_resume)):

    query = resume_embeddings[i].reshape(1, -1)

    distances, indices = index.search(
        np.array(query).astype("float32"),
        5
    )

    for rank in range(5):

        job = df_jobs.iloc[indices[0][rank]]

        all_matches.append({

            "Resume": df_resume.iloc[i]["filename"],

            "Resume Skills": df_resume.iloc[i]["skills"],

            "Job Title": job["job_title"],

            "Company": job["company"],

            "Location": job["job_location"],

            "Required Skills": job["job_skills"],

            "Similarity Score": distances[0][rank]

        })

In [ ]:
matched_jobs = pd.DataFrame(all_matches)

matched_jobs.head()

,Resume,Resume Skills,Job Title,Company,Location,Required Skills,Similarity Score
0,KY BA PM UPDATED RESUME .docx,"['C', 'SQL']",Senior Software Engineer (multiple position),GE Digital,"Foxborough, MA","Software Engineering, Scrum, Agile, TDD, BDD, ...",0.598232
1,KY BA PM UPDATED RESUME .docx,"['C', 'SQL']",Senior Application Software Engineer (Web Serv...,Get It Recruit - Information Technology,"Beverly Hills, MI","Java, C#, Node.js, MongoDB, React, Software De...",0.632962
2,KY BA PM UPDATED RESUME .docx,"['C', 'SQL']",Software Engineer,DUAL North America,"Charlotte, NC","Software Development Life Cycle (SDLC), Softwa...",0.634471
3,KY BA PM UPDATED RESUME .docx,"['C', 'SQL']",Principal Software Engineer,"AvidXchange, Inc.","Charlotte, NC",".NET Core, .NET Framework 4.5+, C#, JavaScript...",0.636203
4,KY BA PM UPDATED RESUME .docx,"['C', 'SQL']",Senior Software Engineer,"The Timberline Group, LLC","St Louis, MO","Software Engineering, Agile Scrum SDLC, Integr...",0.643244


In [ ]:
matched_jobs.to_csv(
    os.path.join(output_path, "matched_jobs.csv"),
    index=False
)

print("matched_jobs.csv saved successfully!")

matched_jobs.csv saved successfully!


In [ ]:
print("Total Recommendations:", len(matched_jobs))

matched_jobs.sample(10)

['Resume', 'Resume Skills', 'Job Title', 'Company', 'Location', 'Required Skills', 'Similarity Score']


In [ ]:
print(matched_jobs.head())

                          Resume Resume Skills  \
0  KY BA PM UPDATED RESUME .docx  ['C', 'SQL']   
1  KY BA PM UPDATED RESUME .docx  ['C', 'SQL']   
2  KY BA PM UPDATED RESUME .docx  ['C', 'SQL']   
3  KY BA PM UPDATED RESUME .docx  ['C', 'SQL']   
4  KY BA PM UPDATED RESUME .docx  ['C', 'SQL']   

                                           Job Title  \
0       Senior Software Engineer (multiple position)   
1  Senior Application Software Engineer (Web Serv...   
2                                  Software Engineer   
3                        Principal Software Engineer   
4                           Senior Software Engineer   

                                   Company           Location  \
0                               GE Digital     Foxborough, MA   
1  Get It Recruit - Information Technology  Beverly Hills, MI   
2                       DUAL North America      Charlotte, NC   
3                        AvidXchange, Inc.      Charlotte, NC   
4                The Timberline Group